# Module 6 · HTTP & API Fundamentals for AI Engineers

**Purpose:** Every AI system in production is accessed through HTTP APIs. This notebook teaches the protocol, status codes, authentication, and patterns you will use in EVERY backend notebook.

**Why this exists:** Notebooks 11-13 build FastAPI servers, stream SSE tokens, and implement MCP over HTTP. Without understanding HTTP, those notebooks are just magic incantations.

---

## 1 -- What is HTTP? (The Language of the Web)

HTTP (HyperText Transfer Protocol) is how computers talk to each other over the internet. Every API call, every webpage, every LLM query uses HTTP.

### The Request-Response Cycle

```
CLIENT (your app)                     SERVER (API provider)
     |                                    |
     |--- HTTP Request ------------------>|
     |    Method: POST                    |
     |    URL: /v1/chat/completions       |
     |    Headers: Authorization: sk-...  |
     |    Body: {"messages": [...]}       |
     |                                    |
     |<-- HTTP Response ------------------|
     |    Status: 200 OK                  |
     |    Headers: Content-Type: json     |
     |    Body: {"content": "Hello!"}     |
```

### HTTP Methods (Verbs)

| Method | Purpose | Analogy | AI Example |
|--------|---------|---------|------------|
| **GET** | Read data (no changes) | Looking at a menu | Get model health status |
| **POST** | Create/send data | Placing an order | Send prompt to LLM |
| **PUT** | Replace entirely | Rewriting your order | Update model config |
| **DELETE** | Remove | Canceling your order | Delete a deployment |
| **PATCH** | Update partially | Adding a side dish | Change one setting |

### What is a URL?

```
https://api.openai.com:443/v1/chat/completions?model=gpt-4o
|       |              |   |                    |
scheme  host          port  path                query parameter
```

- **localhost:8000** = your own computer, port 8000
- **Ports** are like apartment numbers. One computer can run many services on different ports.


## 📑 Table of Contents

* [1 -- What is HTTP? (The Language of the Web)](#1-what-is-http-the-language-of-the-web)
  * [The Request-Response Cycle](#the-request-response-cycle)
  * [HTTP Methods (Verbs)](#http-methods-verbs)
  * [What is a URL?](#what-is-a-url)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [2 -- Status Codes: What the Server Tells You](#2-status-codes-what-the-server-tells-you)
  * [The 5 Categories](#the-5-categories)
  * [The Status Codes You WILL Encounter Daily](#the-status-codes-you-will-encounter-daily)
  * [429 Is Your Best Friend (and Worst Enemy)](#429-is-your-best-friend-and-worst-enemy)
* [3 -- Headers & Authentication](#3-headers-authentication)
  * [What Are Headers?](#what-are-headers)
  * [Authentication Methods for AI APIs](#authentication-methods-for-ai-apis)
* [4 -- Environment Variables: Never Hardcode Secrets](#4-environment-variables-never-hardcode-secrets)
  * [The #1 Security Rule in AI Engineering](#the-1-security-rule-in-ai-engineering)
  * [What Is an Environment Variable?](#what-is-an-environment-variable)
  * [The .env File Pattern (Most Common in Production)](#the-env-file-pattern-most-common-in-production)
* [5 -- What Are REST APIs and Load Balancers?](#5-what-are-rest-apis-and-load-balancers)
  * [REST API = Organized URLs for Your AI Service](#rest-api-organized-urls-for-your-ai-service)
  * [What is A Load Balancer?](#what-is-a-load-balancer)
  * [What is SSE? (Server-Sent Events)](#what-is-sse-server-sent-events)
* [Knowledge Check](#knowledge-check)
  * [Question 1: HTTP Methods](#question-1-http-methods)
  * [Question 2: Status Code 429](#question-2-status-code-429)
  * [Question 3: API Key Security](#question-3-api-key-security)
  * [Question 4: SSE vs REST](#question-4-sse-vs-rest)
* [Practical Practice](#practical-practice)
  * [Exercise 1: API Design](#exercise-1-api-design)
  * [Exercise 2: Error Handling](#exercise-2-error-handling)
  * [Exercise 3: Environment Variables](#exercise-3-environment-variables)
* [Summary](#summary)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Every model, RAG system, or Agent you build is ultimately exposed as an API. If you don't understand REST, JSON, and status codes, your brilliant AI cannot be used by the frontend engineers.

**Mental Model:** An API is a restaurant waiter. The frontend (customer) asks the waiter for Data. The waiter takes the request to the backend (kitchen/model) and returns the JSON meal.

**Common Junior Pitfall:** Returning generic `200 OK` status codes even when an error occurs, burying the error in the JSON payload and breaking standard client error handling.

---


In [ ]:
# Cell 1 -- Making HTTP requests with Python
import json

# The 'requests' library is the standard way to make HTTP calls in Python
# In production, you'd use: import requests
# Here we simulate to avoid network dependency

class HTTPSimulator:
    def get(self, url, headers=None):
        print(f'GET {url}')
        print(f'  Headers: {headers}')
        return {'status': 200, 'body': {'status': 'healthy', 'model_loaded': True}}

    def post(self, url, headers=None, json_body=None):
        print(f'POST {url}')
        print(f'  Headers: {headers}')
        print(f'  Body: {json.dumps(json_body, indent=2)[:200]}')
        return {'status': 200, 'body': {'content': 'Hello! I am an AI assistant.', 'usage': {'tokens': 42}}}

http = HTTPSimulator()

# Example 1: GET request (check if service is running)
print('--- Example 1: Health Check (GET) ---')
response = http.get('http://localhost:8000/health')
print(f'  Response: {response["status"]} -> {response["body"]}\n')

# Example 2: POST request (send prompt to LLM)
print('--- Example 2: LLM Inference (POST) ---')
response = http.post(
    'https://api.openai.com/v1/chat/completions',
    headers={'Authorization': 'Bearer sk-...', 'Content-Type': 'application/json'},
    json_body={'model': 'gpt-4o', 'messages': [{'role': 'user', 'content': 'Hello'}]}
)
print(f'  Response: {response["status"]} -> {response["body"]}')


---
## 2 -- Status Codes: What the Server Tells You

### The 5 Categories

| Range | Category | Meaning | Your Response |
|:---:|---|---|---|
| **2xx** | Success | Request worked | Process the result |
| **3xx** | Redirect | Go somewhere else | Follow the redirect |
| **4xx** | Client error | YOUR request was wrong | Fix your request |
| **5xx** | Server error | THEIR server broke | Retry later |

### The Status Codes You WILL Encounter Daily

| Code | Name | What It Means | AI Engineering Context |
|:---:|---|---|---|
| **200** | OK | Success | Model returned prediction |
| **201** | Created | Resource created | New deployment registered |
| **400** | Bad Request | Your input is malformed | Invalid prompt format |
| **401** | Unauthorized | Missing/wrong API key | Check `Authorization` header |
| **403** | Forbidden | Valid key but no permission | API key doesn't have access to GPT-4o |
| **404** | Not Found | URL doesn't exist | Wrong endpoint path |
| **422** | Unprocessable Entity | Valid JSON but bad values | temperature=5.0 (max is 2.0) |
| **429** | Too Many Requests | Rate limit exceeded | **THE most common LLM error** -> retry with backoff |
| **500** | Internal Server Error | Server crashed | Their bug, not yours |
| **503** | Service Unavailable | Server overloaded | **Used in load shedding (NB25)** |

### 429 Is Your Best Friend (and Worst Enemy)

Every LLM provider rate-limits you. When you hit the limit:
1. You get `429 Too Many Requests`
2. Response includes `Retry-After: 30` header (wait 30 seconds)
3. Circuit breaker (NB25) automatically handles this


In [ ]:
# Cell 2 -- Handling status codes (production pattern)
import time, random

def call_api_with_retry(url, max_retries=3, base_delay=1.0):
    """Production-grade API call with exponential backoff."""
    for attempt in range(max_retries):
        # Simulate API response (sometimes fails)
        status = random.choice([200, 200, 200, 429, 500])

        if status == 200:
            print(f'  Attempt {attempt+1}: 200 OK -> Success!')
            return {'content': 'API response here'}
        elif status == 429:
            delay = base_delay * (2 ** attempt)  # Exponential backoff: 1s, 2s, 4s
            print(f'  Attempt {attempt+1}: 429 Rate Limited -> waiting {delay:.0f}s')
            # time.sleep(delay)  # In production: actually wait
        elif status == 500:
            print(f'  Attempt {attempt+1}: 500 Server Error -> retrying...')
        else:
            print(f'  Attempt {attempt+1}: {status} -> Not retryable, failing.')
            return None

    print(f'  All {max_retries} attempts failed.')
    return None

random.seed(42)
print('API Call with Retry Pattern:')
result = call_api_with_retry('https://api.openai.com/v1/completions')
print(f'\nThis pattern is used in:')
print(f'  - NB25: AI Circuit Breakers (automated retry + fallback)')
print(f'  - NB31: MCP tool invocations')
print(f'  - NB15: LLM streaming endpoints')


---
## 3 -- Headers & Authentication

### What Are Headers?

Headers are **metadata** attached to every HTTP request/response. They tell the server WHO you are and WHAT format you expect.

| Header | Purpose | Example |
|--------|---------|--------|
| `Authorization` | Prove your identity | `Bearer sk-abc123...` |
| `Content-Type` | Format of request body | `application/json` |
| `Accept` | Format you want back | `text/event-stream` (for SSE in NB15) |
| `Retry-After` | Server says when to retry | `30` (seconds) |

### Authentication Methods for AI APIs

| Method | Security | Used By |
|--------|:---:|---|
| **API Key** in header | Medium | OpenAI, Anthropic, most LLMs |
| **OAuth2 Bearer Token** | High | Google Cloud, enterprise APIs |
| **mTLS** (mutual TLS) | Very high | Internal microservices |


---
## 4 -- Environment Variables: Never Hardcode Secrets

### The #1 Security Rule in AI Engineering

**NEVER put API keys in your code.** Use environment variables instead.

```python
# BAD (key in code, leaked if pushed to GitHub):
api_key = 'sk-abc123secretkey456'

# GOOD (key in environment, never in code):
import os
api_key = os.environ['OPENAI_API_KEY']
```

### What Is an Environment Variable?

It's a named value stored in your operating system's memory (not in your code files).

```bash
# Set an environment variable (Bash/Linux/Mac):
export OPENAI_API_KEY='sk-abc123'

# Set an environment variable (Windows PowerShell):
$env:OPENAI_API_KEY = 'sk-abc123'

# Set for a single command:
OPENAI_API_KEY='sk-abc123' python my_app.py
```

### The .env File Pattern (Most Common in Production)

```bash
# .env file (NEVER commit to git, add to .gitignore!)
OPENAI_API_KEY=sk-abc123
WANDB_API_KEY=wandb-xyz789
MLFLOW_TRACKING_URI=http://mlflow.internal:5000
DATABASE_URL=postgresql://user:pass@db:5432/mydb
```

```python
# In Python: load .env file automatically
from dotenv import load_dotenv
load_dotenv()  # Reads .env file into os.environ
key = os.environ['OPENAI_API_KEY']  # Now available!
```


In [ ]:
# Cell 3 -- Environment variables demo
import os

# Demonstrate reading env vars (with safe fallbacks)
def get_config():
    """Production config pattern: read from env, fail loudly if missing."""
    config = {
        'openai_key': os.environ.get('OPENAI_API_KEY', 'NOT SET'),
        'wandb_key': os.environ.get('WANDB_API_KEY', 'NOT SET'),
        'mlflow_uri': os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5000'),
        'debug_mode': os.environ.get('DEBUG', 'false').lower() == 'true',
    }
    # Check for required keys
    required = ['openai_key', 'wandb_key']
    missing = [k for k in required if config[k] == 'NOT SET']
    if missing:
        print(f'  WARNING: Missing env vars: {missing}')
        print(f'  Set them with: export OPENAI_API_KEY=sk-your-key')
    return config

print('Environment Variable Configuration:')
config = get_config()
for k, v in config.items():
    display_v = '***' + str(v)[-4:] if 'key' in k and v != 'NOT SET' else v
    print(f'  {k}: {display_v}')

print(f'\nUsed everywhere in the curriculum:')
print(f'  NB12: WANDB_API_KEY for experiment tracking')
print(f'  NB16: API keys for model serving auth')
print(f'  NB31: MCP server credentials')
print(f'  NB33: LITELLM_API_KEY for multi-provider routing')


---
## 5 -- What Are REST APIs and Load Balancers?

### REST API = Organized URLs for Your AI Service

```
GET  /health                    -> Check if service is alive
POST /v1/chat/completions       -> Send prompt, get response
GET  /v1/models                 -> List available models
POST /v1/embeddings             -> Generate embeddings
DELETE /v1/fine-tunes/{id}      -> Cancel a fine-tuning job
```

These are the actual OpenAI API endpoints. FastAPI (NB15) and BentoML (NB16) let you build yours.

### What is A Load Balancer?

When millions of users hit your API, one server can't handle it all:

```
                          /--> Server 1 (GPU: A100)
Users --> Load Balancer --|--> Server 2 (GPU: A100)
                          \--> Server 3 (GPU: A100)
```

The load balancer distributes requests evenly (round-robin, least-connections, etc.).

**In Kubernetes (NB16):** The load balancer is built in. When you create a `Service`, K8s automatically routes traffic across all your pods.

### What is SSE? (Server-Sent Events)

For LLM streaming (NB15), the response arrives token by token:

```
POST /v1/chat/completions (stream=true)

Response (Content-Type: text/event-stream):
data: {"content": "Hello"}

data: {"content": " world"}

data: {"content": "!"}

data: [DONE]
```

Each `data:` line is one token. The browser renders them as they arrive (no waiting for full response).


In [ ]:
# Cell 4 -- SSE Streaming Simulation
# This simulates what happens when you call an LLM with stream=True
import time

def simulate_sse_stream(prompt):
    """Simulate Server-Sent Events streaming from an LLM."""
    response_tokens = ['Hello', '!', ' I', ' am', ' an', ' AI', ' assistant', '.',
                       ' How', ' can', ' I', ' help', ' you', ' today', '?']
    
    print(f'POST /v1/chat/completions (stream=true)')
    print(f'  Prompt: "{prompt}"')
    print(f'  Response (Content-Type: text/event-stream):')
    print()
    
    full_response = ''
    for i, token in enumerate(response_tokens):
        # Each SSE event is a JSON object with the token
        event = f'data: {{"id": "chunk-{i}", "content": "{token}"}}'
        print(f'  {event}')
        full_response += token
        # time.sleep(0.05)  # In real streaming, tokens arrive with latency
    
    print(f'  data: [DONE]')
    print(f'\nFull response assembled: "{full_response}"')
    print(f'Tokens received: {len(response_tokens)}')
    print(f'\nKey insight: User sees "Hello!" immediately, not after the full response.')
    print(f'This is Time to First Token (TTFT) -- a critical LLM performance metric.')

simulate_sse_stream('Hello, who are you?')


---
## Knowledge Check

### Question 1: HTTP Methods
You want to update a user's email address without changing any other fields. Which HTTP method should you use: PUT or PATCH? Why?

<details>
<summary>Click to reveal answer</summary>

**PATCH.** PUT replaces the ENTIRE resource -- if you PUT with only the email field, all other fields would be lost (or you'd need to include them all). PATCH updates only the specified fields, leaving everything else unchanged. For updating a single field, PATCH is the correct and efficient choice.
</details>

### Question 2: Status Code 429
Your LLM API calls keep returning 429. What is happening and what is the correct retry strategy?

<details>
<summary>Click to reveal answer</summary>

429 = **Too Many Requests** (rate limit exceeded). The correct strategy is **exponential backoff**: retry after 1s, then 2s, then 4s, etc. Check the `Retry-After` header for the exact wait time. Do NOT retry immediately in a loop -- that makes the problem worse. Add jitter (random delay) to avoid all clients retrying at the same time.
</details>

### Question 3: API Key Security
A junior developer committed an OpenAI API key to GitHub. The key is in the git history even after they deleted the file. What should they do?

<details>
<summary>Click to reveal answer</summary>

**Immediately revoke the API key** on the OpenAI dashboard -- this is the #1 priority. Generate a new key and store it in environment variables (never in code). The old key is compromised: git history is permanent, and bots scan GitHub for exposed keys within seconds. Also add `.env` to `.gitignore` and consider using `git-secrets` or `gitleaks` to prevent future leaks.
</details>

### Question 4: SSE vs REST
Why do LLM APIs use SSE (Server-Sent Events) instead of a regular REST response?

<details>
<summary>Click to reveal answer</summary>

LLMs generate tokens one at a time (autoregressive). A regular REST response waits until ALL tokens are generated before sending anything (high latency). SSE sends each token as it's generated, so the user sees the first word within ~100ms instead of waiting 5-10 seconds for the complete response. This dramatically improves the user experience (perceived speed).
</details>


---
## Practical Practice

### Exercise 1: API Design
Design the REST API endpoints (method + URL + request body + response) for an AI image generation service with: generate image, get image status, list all images, delete an image.

### Exercise 2: Error Handling
Write a Python function `safe_api_call(url, api_key)` that makes a POST request, handles status codes 200, 401, 429, and 500 appropriately, and implements exponential backoff for retriable errors.

### Exercise 3: Environment Variables
Write a `Config` class using the `.env` pattern that loads `OPENAI_API_KEY`, `MODEL_NAME` (default: 'gpt-4o'), and `MAX_TOKENS` (default: 1000, must be int). Raise an error if the API key is missing.


In [ ]:
# ===== EXERCISE SOLUTIONS =====
import os

print('Ex 1: Image Generation API Design')
endpoints = [
    ('POST', '/v1/images/generate', '{"prompt": "...", "size": "1024x1024"}', '201 Created + {"id": "img_abc", "status": "processing"}'),
    ('GET', '/v1/images/{id}', 'None', '200 OK + {"id": "img_abc", "status": "completed", "url": "..."}'),
    ('GET', '/v1/images', 'None (query: ?limit=20)', '200 OK + {"images": [...], "total": 42}'),
    ('DELETE', '/v1/images/{id}', 'None', '204 No Content'),
]
for method, url, body, response in endpoints:
    print(f'  {method:6} {url:25} -> {response[:60]}')

print('\nEx 2: Safe API Call with Retry')
print('''def safe_api_call(url, api_key, max_retries=3):
    import time, random
    for attempt in range(max_retries):
        response = requests.post(url, headers={"Authorization": f"Bearer {api_key}"})
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 401:
            raise ValueError("Invalid API key")
        elif response.status_code == 429:
            wait = (2 ** attempt) + random.uniform(0, 1)  # exponential + jitter
            time.sleep(wait)
        elif response.status_code >= 500:
            time.sleep(2 ** attempt)
    raise TimeoutError(f"Failed after {max_retries} retries")''')

print('\nEx 3: Config Class')
class Config:
    def __init__(self):
        self.api_key = os.environ.get('OPENAI_API_KEY')
        if not self.api_key:
            raise ValueError('OPENAI_API_KEY environment variable is required!')
        self.model_name = os.environ.get('MODEL_NAME', 'gpt-4o')
        self.max_tokens = int(os.environ.get('MAX_TOKENS', '1000'))

# Demo (won't crash since we handle the missing key case)
try:
    config = Config()
except ValueError as e:
    print(f'  Expected error: {e}')
    print(f'  Fix: export OPENAI_API_KEY=sk-your-key-here')


---
## Summary

| Concept | What You Learned | Where It Appears |
|---------|-----------------|------------------|
| HTTP request/response | How computers communicate | Every API call |
| Methods (GET, POST) | Read vs write operations | NB15, 12, 13 |
| Status codes | 200 OK, 429 Rate Limit, 503 Overloaded | NB25 (circuit breakers) |
| Headers | Authentication, content type | NB15, 12, 13 |
| Environment variables | Store secrets safely | NB12, 12, 13, 17 |
| .env files | Config management | All production code |
| REST APIs | Organized URL patterns | NB15, 12 |
| Load balancers | Distribute traffic | NB16 (K8s) |
| SSE | Token-by-token streaming | NB15 |

**Next →** `29_async_serving_streaming.ipynb` — Module 2.1 -- High-Performance Async Serving & LLM Token Streaming